In [ ]:
%pip install pymongo mysql-connector-python
import mysql.connector
from pymongo import MongoClient
import json
import os

mysql_conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="saludusm"
)
mysql_cursor= mysql_conn.cursor(dictionary=True)

mongo_url= "mongodb://mongo1:30001,mongo2:30002,mongo3:30003/?replicaSet=my-replica-set"
mongo_client= MongoClient(mongo_url)

db_mongo= mongo_client["salud_usm"]

print("conexion establecida correctamente")
print("Nodos en el replica set:", mongo_client.nodes)


In [ ]:
# migrar catalogos
catalogos = ['prevision', 'especialidad', 'estado_cita', 'centro_medico']
for tabla in catalogos:
    mysql_cursor.execute(f"SELECT * FROM {tabla}")
    datos = mysql_cursor.fetchall()
    if datos:
        db_mongo[tabla].drop() #Limpiar si ya existe la coleccion
        db_mongo[tabla].insert_many(datos)
        print(f"Migrados {len(datos)} registros a la colección '{tabla}'.")
# migrar pacientes
mysql_cursor.execute("""
SELECT p.id_paciente, u.rut, u.nombre, u.apellido, u.email, u.telefono, u.sexo, p.id_prevision
FROM paciente p JOIN usuario u ON p.email = u.email
""")
pacientes = mysql_cursor.fetchall()
if pacientes:
    db_mongo.paciente.drop()
    db_mongo.paciente.insert_many(pacientes)
    print(f"Migrados {len(pacientes)} pacientes.")

# migrar medicos
mysql_cursor.execute("""
SELECT m.id_medico, u.rut, u.nombre, u.apellido, u.email, u.telefono, u.sexo
FROM medico m JOIN usuario u ON m.email = u.email
""")
medicos = mysql_cursor.fechall()
for m in medicos:
    mysql_cursor.execute(f"SELECT id_especialidad FROM medico_especialidad WHERE id_medico = {m['id_medico']}")
    m['id_especialidades'] = [e['id_especialidad'] for e in mysql_cursor.fetchall()]
if medicos:
    db_mongo.medico.drop()
    db_mongo.medico.insert_many(medicos)
    print(f"Migrados {len(medicos)} médicos.")

# Migrar citas (doc central con Atención, Diagnosticos y recetas)
mysql_cursor.execute("SELECT * FROM cita")
citas = mysql_cursor.fetchall()
for c in citas:
    mysql_cursor.execute(f"SELECT id_atencion, motivo, observaciones FROM atencion WHERE id_cita = {c['id_cita']}")
    atencion = mysql_cursor.fetchone()
    if atencion:
        # Anidar diagnosticos
        mysql_cursor.execute(f"""
            SELECT d.id_diagnostico, d.descripcion 
            FROM diagnostico_atencion da 
            JOIN diagnostico d ON da.id_diagnostico = d.id_diagnostico 
            WHERE da.id_atencion = {atencion['id_atencion']}
        """)
        atencion['diagnosticos'] = mysql_cursor.fetchall()
        
        # Anidar recetas
        mysql_cursor.execute(f"""
            SELECT m.id_medicamento, m.nombre as nombre_medicamento, r.dosis, r.dias_tratamiento 
            FROM receta_linea r 
            JOIN medicamento m ON r.id_medicamento = m.id_medicamento 
            WHERE r.id_atencion = {atencion['id_atencion']}
        """)
        atencion['recetas'] = mysql_cursor.fetchall()
        
        c['atencion'] = atencion

if citas:
    db_mongo.cita.drop()
    db_mongo.cita.insert_many(citas)
    print(f"Migradas {len(citas)} citas con sus atenciones anidadas.")




In [ ]:
#esto creará la carpeta JSON si no existe
carpeta_json = 'JSON'
os.makedirs(carpeta_json, exist_ok=True)
colecciones = db_mongo.list_collection_names()
for col in colecciones:
    datos = list(db_mongo[col].find({}, {'_id': 0}))
    ruta_archivo = os.path.join(carpeta_json, f"{col}.json")
    with open(ruta_archivo, 'w', encoding='utf-8') as f:
        json.dump(datos, f, ensure_ascii=False, indent=4, default=str)

print(f"Archivos JSON generados en la carpeta '{carpeta_json}'.")